# Configure the client

The `PrescientClient` needs your Organization host and your API key. It exchanges the API key with your Organization's Keycloak realm for a short-lived access token, and gets a new token before the old one expires.

| Setting | Required | Value |
| --- | --- | --- |
| `prescient_endpoint_url` | Yes | Your Organization host, for example `https://<org>.<env>.prescient.earth` |
| `prescient_api_key` | Yes | Your API key (`pk_...`). Create it in the Keycloak Account Console of your Organization. See "Create an API key" in [Migrate to version 1.0](../migration.md) |
| `prescient_auth_url` | No | The Keycloak base URL. See "Set the Keycloak URL" below |

You can give the settings to the client in three ways:

1. Make a `Settings` object.
2. Put the settings in an env file.
3. Set environment variables.

Environment variables have the highest precedence. A value in an environment variable overrides the same value in an env file.

Keep your API key secret. Do not type it into a notebook that you share or commit. Use an env file that is not under version control, or an environment variable.

A setting from a version before 1.0 causes a `ValueError` that names the setting. See [Migrate to version 1.0](../migration.md).

## 1. Make a `Settings` object

Values in a `Settings` object override values from environment variables and env files. This example reads the API key from an environment variable, so the key is not in the notebook.

In [ ]:
import os

from prescient_sdk import PrescientClient, Settings

settings = Settings(
    prescient_endpoint_url="https://<org>.<env>.prescient.earth",
    prescient_api_key=os.environ["MY_PRESCIENT_API_KEY"],
)

client = PrescientClient(settings=settings)

## 2. Use an env file

Put each setting on one line, with the name in capital letters:

```
PRESCIENT_ENDPOINT_URL="https://<org>.<env>.prescient.earth"
PRESCIENT_API_KEY="pk_..."
```

If the file is named `config.env` and is in the working directory, the client reads it automatically. For a file at a different path, give the path to the client:

In [ ]:
client = PrescientClient(env_file="/path/to/config.env")

## 3. Set environment variables

Use capital letters for the names. For example, in a bash shell:

```
export PRESCIENT_ENDPOINT_URL="https://<org>.<env>.prescient.earth"
export PRESCIENT_API_KEY="pk_..."
```

Then make the client with no arguments:

In [ ]:
client = PrescientClient()

## Set the Keycloak URL

The client finds the Keycloak server from `prescient_endpoint_url`:

- If the host name contains `staging`, it uses `https://keycloak.staging.prescient.earth`.
- For all other hosts, it uses `https://keycloak.prescient.earth`.

The Keycloak realm is the first part of the host name. For `https://spk-sandbox.staging.prescient.earth`, the realm is `spk-sandbox`.

Set `prescient_auth_url` only if your Keycloak server is at a different URL:

In [ ]:
settings = Settings(
    prescient_endpoint_url="https://<org>.<env>.prescient.earth",
    prescient_api_key=os.environ["MY_PRESCIENT_API_KEY"],
    prescient_auth_url="https://keycloak.example.com",
)

## Check the connection

The client exchanges the API key the first time it needs a token. If Keycloak rejects the key, the client raises `PrescientAuthError`. Create a new key if yours is expired or revoked.

In [ ]:
from prescient_sdk import PrescientAuthError

try:
    response = client.session.get(client.stac_url)
    response.raise_for_status()
except PrescientAuthError as error:
    print(f"Sign-in failed: {error}")